# Model A: Original-Image Transfer-Learning Baseline

This notebook trains Model A on the original 128×128 images. The committed training and validation split files are used for optimization and model selection. The untouched test split is intentionally not loaded until the final evaluation stage.

## Environment

Run the data-exploration notebook first when using a new Colab runtime. It mounts Drive, clones the public repository, restores the dataset, and creates the reproducible split CSVs.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml

GITHUB_REPO_URL = 'https://github.com/shaverm96/DSBA-6165-Sum-2026-Take-Home-Exam.git'
COLAB_REPO_DIR = Path('/content/Applied-AI-Midterm')

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if (COLAB_REPO_DIR / '.git').exists():
        pull_result = subprocess.run(
            ['git', 'pull', '--ff-only', 'origin', 'main'],
            cwd=COLAB_REPO_DIR,
            capture_output=True,
            text=True,
        )
        if pull_result.returncode != 0:
            raise RuntimeError(f'Could not update the Colab repository: {pull_result.stderr.strip()}')
        print(pull_result.stdout.strip() or 'Colab repository is already up to date.')
    else:
        clone_result = subprocess.run(
            ['git', 'clone', GITHUB_REPO_URL, str(COLAB_REPO_DIR)],
            capture_output=True,
            text=True,
        )
        if clone_result.returncode != 0:
            raise RuntimeError(f'Could not clone the repository: {clone_result.stderr.strip()}')
        print('Cloned the latest public repository.')
    PROJECT_ROOT = COLAB_REPO_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_source = PROJECT_ROOT / 'src' / 'models' / 'classifier.py'
if not required_source.exists():
    raise FileNotFoundError(
        f'Model A source is missing from {PROJECT_ROOT}. Pull the latest repository commit first.'
    )

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'config.yaml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Configuration file not found: {CONFIG_PATH}')

with CONFIG_PATH.open('r', encoding='utf-8') as stream:
    CONFIG = yaml.safe_load(stream)

from src.models.classifier import ResNet18Classifier
from src.training.train_classifier import (
    build_classification_loaders,
    train_from_config,
)
from src.utils.device import get_device
from src.utils.reproducibility import set_seed

set_seed(int(CONFIG['seed']))
DEVICE = get_device()
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()}')

ModuleNotFoundError: No module named 'src.models'

In [ ]:
TRAINING_CONFIG = CONFIG['training']
DATASET_CONFIG = CONFIG['dataset']
SPLIT_DIR = PROJECT_ROOT / CONFIG['paths']['splits_dir']

train_loader, validation_loader = build_classification_loaders(
    split_dir=SPLIT_DIR,
    class_names=DATASET_CONFIG['classes'],
    image_size=int(TRAINING_CONFIG['image_size']),
    batch_size=int(TRAINING_CONFIG['batch_size']),
    num_workers=int(TRAINING_CONFIG['num_workers']),
)

images, labels, image_paths = next(iter(train_loader))
assert tuple(images.shape[1:]) == (3, 128, 128), images.shape
assert labels.ndim == 1, labels.shape
assert len(image_paths) == images.shape[0]

print(f'Train examples: {len(train_loader.dataset):,}')
print(f'Validation examples: {len(validation_loader.dataset):,}')
print(f'Batch image shape: {tuple(images.shape)}')
print(f'Batch label shape: {tuple(labels.shape)}')
print('The test split was not loaded.')

In [ ]:
CLASSIFIER_CONFIG = CONFIG['model']['classifier']
model = ResNet18Classifier(
    pretrained=bool(CLASSIFIER_CONFIG['pretrained']),
    dropout=float(CLASSIFIER_CONFIG['classifier_dropout']),
)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(model.classifier)
print(f'Total parameters: {parameter_count:,}')
print(f'Trainable parameters: {trainable_count:,}')

## Train Model A

The first configured epoch trains only the new classification head. The remaining epochs fine-tune the pretrained ResNet18 backbone. The best checkpoint is selected using validation loss only.

In [ ]:
model, history = train_from_config(
    config_path=CONFIG_PATH,
    project_root=PROJECT_ROOT,
)

history_df = pd.DataFrame(history)
history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_df.plot(x='epoch', y=['train_loss', 'validation_loss'], ax=axes[0])
axes[0].set_title('Model A loss')
axes[0].set_ylabel('BCE loss')
history_df.plot(x='epoch', y=['train_accuracy', 'validation_accuracy', 'validation_f1'], ax=axes[1])
axes[1].set_title('Model A validation history')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.show()

## Checkpoint and resume

Training writes `models/checkpoints/model_a/last.pt` and `models/checkpoints/model_a/best.pt`, plus the history JSON under `logs/`. To resume after an interruption, pass the `last.pt` path to `train_from_config` instead of starting a new run.